# Exercise 2 – Task: when does the sky change?

**Goal:** the clip `lake_clip.mp4` shows a rowboat on a mountain lake at sunrise – without speech. Find out **when the colour of the sky changes**: a table with one row per second and the fields `sky` and `oars_in_water`, and the list of moments where the sky changes.

Build it yourself in three steps. Every step can be done by **copying code from `video_extraction_example.ipynb` and adapting it**.

Each step: write your code in the empty cell → run the **check** (✓ go on, ✗ shows what is missing; a `NameError` means the variable does not exist yet). Stuck or out of time? Open the **hint**, then the **solution**.

## Setup

Select the **Python 3.11** kernel. The OpenAI API key comes from your `OPENAI_API_KEY` Codespaces secret. This notebook uses the **OpenAI API** (model `gpt-6-luna`); other providers, e.g. Google Gemini or Anthropic Claude, have their own APIs with different code.

In [ ]:
# Python libraries.
import sys
from typing import Literal

import pandas as pd
from pydantic import BaseModel

# helper.py and checks.py are in the main folder, one level up.
sys.path.append("..")
from helper import DATA, ask, extract_frames, show_images, transcribe, video_info
from checks import check_changes, check_frame_paths, check_frames

## Step 1: frames

Show the info of `lake_clip.mp4` and save one frame per second in the folder `frames_lake` – `extract_frames(video_path, folder="frames_lake")`, so the frames of the example stay untouched. Store the file paths in `paths` and show them.

In [ ]:
# Your code here.


In [ ]:
check_frame_paths(paths, expected=8)

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hint</summary>

Example, section 1 – only with the new video and `folder="frames_lake"`.

</details>

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Solution</summary>

```python
video_path = DATA / "videos" / "lake_clip.mp4"
print(video_info(video_path))

paths = extract_frames(video_path, folder="frames_lake")
show_images(paths)
```

</details>

## Step 2: fields per frame

Ask about every frame in a **fixed format** (like in Exercise 1) and build the DataFrame `frames`: one row per second with the columns `t`, `sky` and `oars_in_water`.

| Field | Type | Instruction for the model |
| --- | --- | --- |
| `sky` | one of `"pale"`, `"light pink"`, `"deep pink"` | How strong is the pink colour of the sky? Pick from the list: pale (hardly any colour), light pink, deep pink. |
| `oars_in_water` | `True` or `False` (`bool`) | Are the oars of the boat in the water? |

In [ ]:
# Your code here.


In [ ]:
check_frames(frames, paths)

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hint</summary>

Example, section 2: the same loop over `t`. Instead of `describe()`, use `ask(PROMPT, FrameRecord, [paths[t]])` with a class like `ImageRecord` in Exercise 1 – and add `row["t"] = t`.

</details>

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Solution</summary>

```python
class FrameRecord(BaseModel):
    sky: Literal["pale", "light pink", "deep pink"]
    oars_in_water: bool


PROMPT = """
This is one frame of a video at sunrise.
How strong is the pink colour of the sky? Pick from the list:
pale (hardly any colour), light pink, deep pink.
Are the oars of the boat in the water?
"""

rows = []
for t in range(len(paths)):
    row = ask(PROMPT, FrameRecord, [paths[t]])
    row["t"] = t
    rows.append(row)

frames = pd.DataFrame(rows)
frames
```

</details>

## Step 3: when does the sky change?

Keep only the seconds where `sky` differs from the second before. `frames["sky"] != frames["sky"].shift()` is `True` exactly there – and for `t = 0`, where the clip starts. Store these rows with the columns `t` and `sky` in `changes`.

In [ ]:
# Your code here.


In [ ]:
check_changes(changes, frames)

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hint</summary>

`.shift()` moves the column down by one row, so every value is compared with the second before. Select the rows with `frames.loc[changed, ["t", "sky"]]`.

</details>

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Solution</summary>

```python
changed = frames["sky"] != frames["sky"].shift()
changes = frames.loc[changed, ["t", "sky"]]
changes
```

</details>

**Extra:** run `transcribe(video_path)` on this clip. It has no speech – what comes back, and what would that mean for a merge like in the example?

## Check your understanding

Answer briefly using the outputs above. Try first, then open the hints.

1. What do you lose with one frame every 2 seconds instead of every second? Look at how many segments Whisper returned and how long each one is.
2. Why do we send 8 frames to the OpenAI API and not all 190 frames of the clip?
3. A frame gets the text of a speech segment when `start ≤ t < end`. Why is the end not included?
4. Compare the transcript with what Ueli says in Swiss German. What did Whisper change?
5. The model describes each frame on its own. What can it therefore not know about the video?

**Your answers:**

1. …
2. …
3. …
4. …
5. …

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hints for self-check</summary>

1. Speech shorter than the gap between two frames, such as «Lueg, Bruno», can fall between frames, so no frame carries it. If Whisper merges it into a longer segment, the timing gets coarse instead. Either way short events get lost, so keep the speech table next to the frames.

2. Every frame costs tokens and time, and neighbouring frames look almost the same. Too few frames miss events, too many cost money – so you have to choose.

3. A frame exactly on the border would otherwise belong to two segments. With [start, end) every point in time belongs to exactly one segment.

4. Whisper often answers in Standard German or misspells dialect words. Compare the transcript with a reference before you use it.

5. It sees no motion, no order and no sound: not what happened before or after, and not how long something lasts. This information comes from our time stamps and from the merge.

</details>

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')